# Ontario Highway Traffic Volume Map — Build Notebook

This is the real, executed notebook behind the [interactive map](../docs/index.html) — not a cleaned-up rewrite. It includes the dead ends and bug fixes as they actually happened (a naive route filter that missed overlapping highway designations, a coordinate reference system dead end, a length-calculation bug from splitting lines, a forgotten highway that needed a mid-project rebuild), because that's a more honest picture of the actual GIS/data engineering work than a tidy linear script would be.

**Stack:** ArcGIS Pro, Python (arcpy, pandas, numpy), Leaflet.js

**Data sources:** Ontario Road Network (ORN) Composite Service, MTO Traffic Volumes 1988-2021, MTO LHRS Base Points

---


**Setup.** Just pointing arcpy at whatever geodatabase this project's already using, and telling it to overwrite outputs without complaining, since I'm going to re-run cells a lot while I figure this out.

In [11]:
import arcpy

aprx = arcpy.mp.ArcGISProject("CURRENT")
arcpy.env.workspace = aprx.defaultGeodatabase
arcpy.env.overwriteOutput = True


**First pass at pulling my 6 highways.** I'm hitting the Ontario Road Network Composite Service directly (it's a live feature service, no static shapefile download available) and filtering on `ROUTE_NUMBER` for an exact match. Spoiler: this filter is too naive and I'll have to fix it a few cells from now.

In [12]:
url = "https://services1.arcgis.com/TJH5KDher0W13Kgo/arcgis/rest/services/Ontario_Road_Network_Composite_Service_GeoHub_View_EN/FeatureServer/5"

where_clause = "ROUTE_NUMBER IN ('400', '401', '403', '404', '410', 'QEW')"

arcpy.conversion.FeatureClassToFeatureClass(
    in_features=url,
    out_path=arcpy.env.workspace,
    out_name="highways_filtered",
    where_clause=where_clause
)

<Result 'C:\\Users\\ajots\\OneDrive\\Documents\\ArcGIS\\Projects\\GTA_Highway_Traffic_Volumes\\GTA_Highway_Traffic_Volumes.gdb\\highways_filtered'>

Quick count check — got back 2003 segments. Reasonable number for six highways, but I don't know yet that a chunk of real segments are missing.

In [13]:
count = arcpy.management.GetCount("highways_filtered")
print(count)

2003


Listing every field on this layer so I know what I'm working with — specifically hunting for the route number column and anything else useful (municipality, road class, etc.) for later.

In [14]:
fields = arcpy.ListFields("highways_filtered")
for f in fields:
    print(f.name, f.type)

OBJECTID OID
Shape Geometry
ROAD_NET_ELEMENT_ID Integer
FULL_STREET_NAME String
ABBREVIATED_STREET_NAME String
ALT_STREET_NAME String
DIRECTIONAL_PREFIX String
STREET_TYPE_PREFIX String
STREET_NAME_BODY String
STREET_TYPE_SUFFIX String
DIRECTIONAL_SUFFIX String
ROAD_CLASS String
ROAD_ELEMENT_TYPE String
L_FIRST_HOUSE_NUMBER Integer
L_LAST_HOUSE_NUMBER Integer
L_HOUSE_NUMBER_STRUCTURE String
L_STANDARD_MUNICIPALITY String
R_FIRST_HOUSE_NUMBER Integer
R_LAST_HOUSE_NUMBER Integer
R_HOUSE_NUMBER_STRUCTURE String
R_STANDARD_MUNICIPALITY String
DIRECTION_OF_TRAFFIC_FLOW String
SPEED_LIMIT Integer
NUMBER_OF_LANES Integer
PAVEMENT_STATUS String
JURISDICTION String
ROUTE_NUMBER String
SHIELD_TYPE String
GEOMETRY_UPDATE_DATETIME Date
EFFECTIVE_DATETIME Date
LENGTH Double
Shape_Length Double


Noticed a gap in the map near Mississauga on the 403, so I'm checking every *distinct* value actually stored in `ROUTE_NUMBER` to see if something weird is going on — like combined route designations that my exact-match filter above would silently skip.

In [15]:
with arcpy.da.SearchCursor("highways_filtered", ["ROUTE_NUMBER"]) as cursor:
    unique_values = set(row[0] for row in cursor)

for v in sorted(unique_values, key=str):
    print(v)

400
401
403
404
410
QEW


Casting a wider net — searching by street name instead of route number, to see if there are more "403" segments out there than my route number filter caught.

In [16]:
url = "https://services1.arcgis.com/TJH5KDher0W13Kgo/arcgis/rest/services/Ontario_Road_Network_Composite_Service_GeoHub_View_EN/FeatureServer/5"

where_clause = "FULL_STREET_NAME LIKE '%403%' OR ALT_STREET_NAME LIKE '%403%'"

arcpy.conversion.FeatureClassToFeatureClass(
    in_features=url,
    out_path=arcpy.env.workspace,
    out_name="check_403_gap",
    where_clause=where_clause
)

count = arcpy.management.GetCount("check_403_gap")
print(count)

492


Comparing the two counts directly: segments where `ROUTE_NUMBER` is exactly `'403'` vs. segments where the street name just mentions 403. Big gap here (144 vs 492) is what confirmed something was wrong with my filter.

In [17]:
with arcpy.da.SearchCursor("highways_filtered", ["ROUTE_NUMBER"], where_clause="ROUTE_NUMBER = '403'") as cursor:
    existing_403 = sum(1 for _ in cursor)

print("Existing 403 segments:", existing_403)
print("Segments found by street name search:", count)

Existing 403 segments: 144
Segments found by street name search: 492


Printing every segment that matched on street name but *doesn't* have `ROUTE_NUMBER = '403'` — trying to see exactly what's stored in that field instead, so I can figure out the real pattern.

In [18]:
with arcpy.da.SearchCursor("check_403_gap", ["ROUTE_NUMBER", "FULL_STREET_NAME", "ROAD_CLASS"]) as cursor:
    for row in cursor:
        if row[0] != '403':
            print(row)

('QEW; 403', 'QUEEN ELIZABETH WAY', 'Freeway')
('QEW; 403', 'QUEEN ELIZABETH WAY', 'Freeway')
('', 'HIGHWAY 403', 'Ramp')
('QEW; 403', 'QUEEN ELIZABETH WAY', 'Freeway')
('407', 'HIGHWAY 403', 'Ramp')
('403; 6', 'HIGHWAY 403', 'Freeway')
('QEW; 403', 'QUEEN ELIZABETH WAY', 'Freeway')
('QEW; 403', 'QUEEN ELIZABETH WAY', 'Freeway')
('', 'HIGHWAY 403', 'Ramp')
('403; 6', 'HIGHWAY 403', 'Freeway')
('', 'HIGHWAY 403 & HIGHWAY 407', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'SR403 SHORE', '')
('', 'SR403 SHORE', '')
('', 'ISLAND 4030 GEORGIAN BAY', '')
(

Same idea but tallied instead of a wall of text — counting how often each weird `ROUTE_NUMBER` value and `ROAD_CLASS` shows up. This is what revealed the real story: combined values like `'QEW; 403'` and a ton of blank-route `Ramp` segments.

In [19]:
from collections import Counter

route_vals = Counter()
road_classes = Counter()

with arcpy.da.SearchCursor("check_403_gap", ["ROUTE_NUMBER", "FULL_STREET_NAME", "ROAD_CLASS"]) as cursor:
    for row in cursor:
        if row[0] != '403':
            route_vals[row[0]] += 1
            road_classes[row[2]] += 1

print("ROUTE_NUMBER values found instead:")
for val, n in route_vals.most_common():
    print(f"  {val!r}: {n}")

print("\nROAD_CLASS values for these segments:")
for val, n in road_classes.most_common():
    print(f"  {val!r}: {n}")

ROUTE_NUMBER values found instead:
  '': 235
  'QEW; 403': 60
  '403; 6': 21
  '403; 24': 18
  '407': 5
  '6': 4
  '403; 410': 2
  '24': 2
  '401': 1

ROAD_CLASS values for these segments:
  'Ramp': 231
  'Freeway': 101
  'Resource / Recreation': 9
  '': 3
  'Arterial': 2
  'Local / Strata': 1
  'Expressway / Highway': 1


**The actual fix.** Instead of an exact match, I'm using `LIKE` to catch combined route values (`'QEW; 403'`, `'403; 6'`, etc.), and restricting to `ROAD_CLASS = 'Freeway'` so I don't pull in unrelated roads that just happen to contain the same digits (there's a literal Fire Route 403 out there, unrelated to the highway).

In [20]:
url = "https://services1.arcgis.com/TJH5KDher0W13Kgo/arcgis/rest/services/Ontario_Road_Network_Composite_Service_GeoHub_View_EN/FeatureServer/5"

target_routes = {"400", "401", "403", "404", "410", "QEW"}

like_clause = " OR ".join([f"ROUTE_NUMBER LIKE '%{r}%'" for r in target_routes])
where_clause = f"({like_clause}) AND ROAD_CLASS = 'Freeway'"

arcpy.conversion.FeatureClassToFeatureClass(
    in_features=url,
    out_path=arcpy.env.workspace,
    out_name="highways_mainline",
    where_clause=where_clause
)

print(arcpy.management.GetCount("highways_mainline"))

2090


Ramps mostly don't carry a route number at all, so I'm matching them separately by street name instead (e.g. "HIGHWAY 403"), and only keeping `ROAD_CLASS = 'Ramp'` this time.

In [21]:
street_names = ["HIGHWAY 400", "HIGHWAY 401", "HIGHWAY 403", "HIGHWAY 404", "HIGHWAY 410", "QUEEN ELIZABETH WAY"]

street_clause = " OR ".join([f"FULL_STREET_NAME LIKE '%{s}%'" for s in street_names])
where_clause = f"({street_clause}) AND ROAD_CLASS = 'Ramp'"

arcpy.conversion.FeatureClassToFeatureClass(
    in_features=url,
    out_path=arcpy.env.workspace,
    out_name="highways_ramps",
    where_clause=where_clause
)

print(arcpy.management.GetCount("highways_ramps"))

2421


Tagging each of the two layers above with a `segment_type` field (Mainline vs Ramp) so I can tell them apart later, then merging them into one layer — this is my "v2" fixed highway layer.

In [22]:
arcpy.management.AddField("highways_mainline", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_mainline", "segment_type", "'Mainline'")

arcpy.management.AddField("highways_ramps", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_ramps", "segment_type", "'Ramp'")

arcpy.management.Merge(
    inputs=["highways_mainline", "highways_ramps"],
    output="highways_filtered_v2"
)

print(arcpy.management.GetCount("highways_filtered_v2"))

4511


Sanity check on the merge — tallying route number + segment type combos to make sure nothing unrelated snuck in through my wider `LIKE` filter. Everything that showed up traced back to real interchanges with our target highways, so I'm calling this clean.

In [23]:
from collections import Counter

with arcpy.da.SearchCursor("highways_filtered_v2", ["ROUTE_NUMBER", "segment_type"]) as cursor:
    tally = Counter((row[0], row[1]) for row in cursor)

for (route, seg_type), n in sorted(tally.items(), key=lambda x: -x[1])[:20]:
    print(route, seg_type, n)

﻿ Ramp 2301
401 Mainline 1180
400 Mainline 254
QEW Mainline 229
403 Mainline 134
404 Mainline 93
QEW; 403 Mainline 60
410 Mainline 55
401 Ramp 40
400; 69 Mainline 35
403; 6 Mainline 21
403; 24 Mainline 18
6 Ramp 13
403 Ramp 10
137 Ramp 7
400 Ramp 6
420 Ramp 6
407 Ramp 5
35; 115 Ramp 5
406 Ramp 5


**Starting on the traffic volume side.** Loading the big MTO AADT CSV for the first time just to see what's actually in it — columns, shape, a first look at the data.

In [24]:
import pandas as pd

csv_path = r"data raw\Traffic_Volumes_1988-2021.csv"
aadt_df = pd.read_csv(csv_path)

print(aadt_df.shape)
print(aadt_df.columns.tolist())
aadt_df.head()

<class 'FileNotFoundError'>: [Errno 2] No such file or directory: 'data raw\\Traffic_Volumes_1988-2021.csv'

Trying to figure out what MTO's internal `Hwy No` codes actually correspond to, by searching location text for my target highway names. Turns out this approach is flawed — it also picks up highways that are just *mentioned* nearby, not the highway itself. Kept for the record, but the next cell is the real fix.

In [ ]:
# Look for rows whose location description mentions our target highways by name,
# and see what Hwy No value those rows actually have
keywords = ["QEW", "401", "400", "403", "404", "410"]

for kw in keywords:
    matches = aadt_df[aadt_df["Location Description"].str.contains(kw, na=False)]
    print(f"--- '{kw}' ---")
    print(matches[["Hwy No", "Hwy No Suffix", "Hwy. Type", "Location Description"]].drop_duplicates().head(10))
    print()

**The real check.** Filtering directly by candidate `Hwy No` values and reading the resulting location descriptions to see if they trace a believable route. This is how I found out the QEW is secretly coded as `Hwy No = 1` — its locations trace Fort Erie → Niagara Falls → Hamilton, which is exactly the QEW.

In [ ]:
# Check each candidate Hwy No directly and see if the locations trace a believable route
for hwy in [1, 400, 401, 403, 404, 410]:
    subset = aadt_df[aadt_df["Hwy No"] == hwy]
    print(f"--- Hwy No = {hwy} ({len(subset)} rows) ---")
    print(subset["Location Description"].drop_duplicates().head(8).tolist())
    print()

Filtering the AADT data down to just my 6 highways using the real codes I just confirmed (`1` for QEW).

In [ ]:
target_hwy_codes = [1, 400, 401, 403, 404, 410]  # 1 = QEW

aadt_filtered = aadt_df[aadt_df["Hwy No"].isin(target_hwy_codes)].copy()

print(aadt_filtered.shape)
print(aadt_filtered["Hwy No"].value_counts())

Checking what years are actually available. Found there's no 2020 (COVID-era data gap, probably) and a bogus `9999` placeholder that needs to be excluded, not treated as a real year.

In [ ]:
print(sorted(aadt_filtered["Year"].unique()))

Picking 2021 as the most recent real year and filtering down to just that.

In [ ]:
aadt_2021 = aadt_filtered[aadt_filtered["Year"] == 2021].copy()
print(aadt_2021.shape)
aadt_2021[["Hwy No", "Location Description", "AADT", "LHRS", "Offset"]].head(10)

Checking the AADT column's data type before trying to do math on it — turned out to be stored as text with comma separators (like `"45,300"`), not a clean number.

In [ ]:
print(aadt_2021["AADT"].dtype)
print(aadt_2021["AADT"].head(10))

First attempt at stripping the commas and converting AADT to an actual float.

In [ ]:
aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)

Redid the cleaning with the `9999` year explicitly excluded too (belt-and-suspenders), plus ran `.describe()` to sanity check the distribution — numbers looked reasonable (min ~8K, max ~480K).

In [ ]:
aadt_2021 = aadt_2021[aadt_2021["Year"] != 9999].copy()  # just in case, belt-and-suspenders

aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)

print(aadt_2021["AADT_clean"].describe())

**Dead end, but a useful one.** Downloaded MTO's official LHRS Routes shapefile hoping it'd let me place AADT stations precisely on the highway. Turned out this file has zero fields for LHRS codes — just one continuous line per highway with a highway number field. Not what I needed.

In [ ]:
lhrs_routes_path = r"data raw\lhrs_routes_july2015\LHRS_Routes_July2015.shp"

arcpy.management.CopyFeatures(lhrs_routes_path, "lhrs_routes")

fields = arcpy.ListFields("lhrs_routes")
for f in fields:
    print(f.name, f.type)

Checking what highway codes this Routes file uses, just to understand it before giving up on it.

In [ ]:
with arcpy.da.SearchCursor("lhrs_routes", ["HWY_2015"]) as cursor:
    unique_hwy = sorted(set(row[0] for row in cursor), key=str)

print(unique_hwy[:30])
print(len(unique_hwy))

Confirming my 6 target highways are actually present in this file (they are) — but as noted, this file doesn't have what I actually need, so I pivot to Base Points next.

In [ ]:
target = ['1', '400', '401', '403', '404', '410']
found = [h for h in unique_hwy if h in target]
print(found)

**The file that actually works.** Loading MTO's LHRS Base Points shapefile instead — this one has `LHRS`, `OFFSET`, and even direct `LATITUDE`/`LONGITUDE` columns. Exactly what's needed to place my AADT stations on the map.

In [ ]:
base_points_path = r"data raw\LHRS_Base_Points_July2015.shp"

arcpy.management.CopyFeatures(base_points_path, "lhrs_base_points")

fields = arcpy.ListFields("lhrs_base_points")
for f in fields:
    print(f.name, f.type)

Checking how many of my 327 AADT stations actually have a matching `(LHRS, Offset)` pair in this Base Points file. First run of this check — see the next several cells for why I had to redo it after a notebook reset.

In [ ]:
with arcpy.da.SearchCursor("lhrs_base_points", ["LHRS", "OFFSET", "HWY"]) as cursor:
    base_keys = set((row[0], row[1]) for row in cursor)

# Build the same key from your AADT 2021 data
aadt_keys = set(zip(aadt_2021["LHRS"], aadt_2021["Offset"]))

matched = aadt_keys & base_keys
print(f"AADT stations: {len(aadt_keys)}")
print(f"Base points: {len(base_keys)}")
print(f"Matched: {len(matched)}")

My notebook's kernel reset at some point (closed ArcGIS Pro, went idle, who knows) and wiped my variables — so I'm reloading the AADT data from scratch here to get back to where I was.

In [ ]:
import pandas as pd

csv_path = r"data raw\Traffic_Volumes_1988-2021.csv"
aadt_df = pd.read_csv(csv_path)

target_hwy_codes = [1, 400, 401, 403, 404, 410]  # 1 = QEW
aadt_filtered = aadt_df[aadt_df["Hwy No"].isin(target_hwy_codes)].copy()

aadt_2021 = aadt_filtered[aadt_filtered["Year"] == 2021].copy()
aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)

print(aadt_2021.shape)

This reload immediately broke with a `FileNotFoundError` — checking what directory Python actually thinks it's in, since my relative path stopped resolving.

In [ ]:
import os
print(os.getcwd())

**Fixing the path issue for real.** Building an absolute path off `aprx.homeFolder` instead of a fragile relative path — this stays correct no matter what the notebook's working directory happens to be.

In [ ]:
import arcpy
import os

aprx = arcpy.mp.ArcGISProject("CURRENT")
project_folder = aprx.homeFolder
print(project_folder)

csv_path = os.path.join(project_folder, "data raw", "Traffic_Volumes_1988-2021.csv")
print(csv_path)
print(os.path.exists(csv_path))  # should print True

Reloading the AADT CSV using the new reliable path.

In [ ]:
import pandas as pd

aadt_df = pd.read_csv(csv_path)

target_hwy_codes = [1, 400, 401, 403, 404, 410]  # 1 = QEW
aadt_filtered = aadt_df[aadt_df["Hwy No"].isin(target_hwy_codes)].copy()

aadt_2021 = aadt_filtered[aadt_filtered["Year"] == 2021].copy()
aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)

print(aadt_2021.shape)

Re-running the join match check now that the data's reloaded — confirmed 326 out of 327 stations matched to Base Points.

In [ ]:
with arcpy.da.SearchCursor("lhrs_base_points", ["LHRS", "OFFSET", "HWY"]) as cursor:
    base_keys = set((row[0], row[1]) for row in cursor)

aadt_keys = set(zip(aadt_2021["LHRS"], aadt_2021["Offset"]))

matched = aadt_keys & base_keys
print(f"AADT stations: {len(aadt_keys)}")
print(f"Base points: {len(base_keys)}")
print(f"Matched: {len(matched)}")

Curious about that one unmatched station, so I'm looking it up directly — turned out to be a single 401 station near Wonderland Rd, probably added to MTO's system sometime after this 2015 Base Points snapshot.

In [ ]:
unmatched = aadt_keys - base_keys
print(unmatched)

# Look up that station's details in your AADT data
lhrs_val, offset_val = list(unmatched)[0]
row = aadt_2021[(aadt_2021["LHRS"] == lhrs_val) & (aadt_2021["Offset"] == offset_val)]
print(row[["Hwy No", "Location Description", "AADT_clean", "LHRS", "Offset"]])

Same as the cell above — just re-ran it, looks like a duplicate from re-executing the cell.

In [25]:
unmatched = aadt_keys - base_keys
print(unmatched)

# Look up that station's details in your AADT data
lhrs_val, offset_val = list(unmatched)[0]
row = aadt_2021[(aadt_2021["LHRS"] == lhrs_val) & (aadt_2021["Offset"] == offset_val)]
print(row[["Hwy No", "Location Description", "AADT_clean", "LHRS", "Offset"]])

{(47828, 0.0)}
       Hwy No  Location Description  AADT_clean   LHRS  Offset
41013     401  WONDERLAND RD IC-180     29300.0  47828     0.0


Building a `(LHRS, Offset) -> AADT` lookup dictionary from my cleaned AADT data, then writing those values onto the Base Points layer as a new `AADT_2021` field.

In [26]:
# Build a lookup dict: (LHRS, Offset) -> AADT_clean
aadt_lookup = {
    (row["LHRS"], row["Offset"]): row["AADT_clean"]
    for _, row in aadt_2021.iterrows()
}

# Add a new field to lhrs_base_points and populate it
arcpy.management.AddField("lhrs_base_points", "AADT_2021", "DOUBLE")

with arcpy.da.UpdateCursor("lhrs_base_points", ["LHRS", "OFFSET", "AADT_2021"]) as cursor:
    for row in cursor:
        key = (row[0], row[1])
        if key in aadt_lookup:
            row[2] = aadt_lookup[key]
            cursor.updateRow(row)

Filtering Base Points down to just my 6 highways, keeping only the ones that actually got an AADT value attached — this becomes `aadt_stations_2021`, my final list of real, located traffic stations.

In [27]:
where_clause = "HWY IN ('1','400','401','403','404','410') AND AADT_2021 IS NOT NULL"

arcpy.management.SelectLayerByAttribute("lhrs_base_points", "NEW_SELECTION", where_clause)
arcpy.management.CopyFeatures("lhrs_base_points", "aadt_stations_2021")

count = arcpy.management.GetCount("aadt_stations_2021")
print(count)

319


Spot-checking a handful of rows to make sure the coordinates and AADT values look sane before building anything else on top of them.

In [28]:
with arcpy.da.SearchCursor("aadt_stations_2021", ["HWY", "LATITUDE", "LONGITUDE", "AADT_2021"]) as cursor:
    for i, row in enumerate(cursor):
        if i < 5:
            print(row)

('401', 43.229106, -80.60205599, 50500.0)
('400', 43.84668588, -79.54847475, 140200.0)
('401', 42.48247219, -81.91116389, 22800.0)
('401', 42.24252476, -82.55214227, 22500.0)
('401', 44.917106, -75.197659, 18500.0)


Pulling out just the Mainline segments (not ramps) from my highway layer — this is what's actually getting split into traffic-tier stretches.

In [29]:
where_clause = "segment_type = 'Mainline'"
arcpy.management.SelectLayerByAttribute("highways_filtered_v2", "NEW_SELECTION", where_clause)
arcpy.management.CopyFeatures("highways_filtered_v2", "highways_mainline_only")

<Result 'C:\\Users\\ajots\\OneDrive\\Documents\\ArcGIS\\Projects\\GTA_Highway_Traffic_Volumes\\GTA_Highway_Traffic_Volumes.gdb\\highways_mainline_only'>

**Splitting the highway at each station point** so different parts of the same highway can eventually get different colors, instead of one flat value for the whole route. Went from 2090 whole segments to 2816 smaller stretches.

In [30]:
arcpy.management.SplitLineAtPoint(
    in_features="highways_mainline_only",
    point_features="aadt_stations_2021",
    out_feature_class="highways_split",
    search_radius="100 Meters"
)

count = arcpy.management.GetCount("highways_split")
print(count)

2816


First attempt at joining each stretch to its nearest station, using a 500m search radius. This left 878 out of 2816 stretches with no match — turns out rural highway stretches can be way more than 500m from the nearest monitoring station.

In [31]:
arcpy.analysis.SpatialJoin(
    target_features="highways_split",
    join_features="aadt_stations_2021",
    out_feature_class="highways_with_aadt",
    join_operation="JOIN_ONE_TO_ONE",
    match_option="CLOSEST",
    search_radius="500 Meters"
)

count = arcpy.management.GetCount("highways_with_aadt")
print(count)

# Quick check: how many got a real AADT value vs came back empty
with arcpy.da.SearchCursor("highways_with_aadt", ["AADT_2021"]) as cursor:
    null_count = sum(1 for row in cursor if row[0] is None)
print(f"Segments with no matched AADT: {null_count}")

2816
Segments with no matched AADT: 878


**Fixed it** by widening the search radius to 50km, which comfortably covers even sparse rural gaps. This got every single stretch matched — 0 unmatched.

In [32]:
arcpy.analysis.SpatialJoin(
    target_features="highways_split",
    join_features="aadt_stations_2021",
    out_feature_class="highways_with_aadt_v2",
    join_operation="JOIN_ONE_TO_ONE",
    match_option="CLOSEST",
    search_radius="50000 Meters"  # 50 km - should comfortably catch even the sparsest rural gaps
)

count = arcpy.management.GetCount("highways_with_aadt_v2")
print(count)

with arcpy.da.SearchCursor("highways_with_aadt_v2", ["AADT_2021"]) as cursor:
    null_count = sum(1 for row in cursor if row[0] is None)
print(f"Segments with no matched AADT: {null_count}")

2816
Segments with no matched AADT: 0


Sanity-checking the AADT distribution on the newly split data, to make sure the split/join process didn't distort anything. Numbers matched the original station-level stats almost exactly.

In [33]:
# Add highways_with_aadt_v2 to your map and visually confirm color/value makes sense once symbolized later.
# For now, just check the AADT distribution looks reasonable across the new stretch-level data:

with arcpy.da.SearchCursor("highways_with_aadt_v2", ["AADT_2021"]) as cursor:
    values = [row[0] for row in cursor]

import numpy as np
values = np.array(values)
print(f"Count: {len(values)}")
print(f"Min: {values.min()}, Max: {values.max()}")
print(f"25th pct: {np.percentile(values, 25):.0f}")
print(f"Median: {np.percentile(values, 50):.0f}")
print(f"75th pct: {np.percentile(values, 75):.0f}")

Count: 2816
Min: 8100.0, Max: 480900.0
25th pct: 29300
Median: 80400
75th pct: 176200


Calculating the tercile break points (33rd/67th percentile) that'll define my Low/Medium/High traffic tiers.

In [34]:
import numpy as np

low_threshold = np.percentile(values, 33.33)
high_threshold = np.percentile(values, 66.67)

print(f"Low/Medium boundary: {low_threshold:.0f}")
print(f"Medium/High boundary: {high_threshold:.0f}")

Low/Medium boundary: 40100
Medium/High boundary: 130100


Actually writing the `traffic_tier` label onto every stretch based on those break points.

In [35]:
arcpy.management.AddField("highways_with_aadt_v2", "traffic_tier", "TEXT", field_length=20)

with arcpy.da.UpdateCursor("highways_with_aadt_v2", ["AADT_2021", "traffic_tier"]) as cursor:
    for row in cursor:
        aadt = row[0]
        if aadt is None:
            row[1] = "No Data"
        elif aadt <= low_threshold:
            row[1] = "Low"
        elif aadt <= high_threshold:
            row[1] = "Medium"
        else:
            row[1] = "High"
        cursor.updateRow(row)

Tallying up how many stretches landed in each tier — came out almost perfectly even (about 940 each), which makes sense since I used exact terciles.

In [36]:
from collections import Counter

with arcpy.da.SearchCursor("highways_with_aadt_v2", ["traffic_tier"]) as cursor:
    tally = Counter(row[0] for row in cursor)

for tier, n in tally.items():
    print(tier, n)

Medium 937
High 938
Low 941


Pulling the Ramp segments back in (they were set aside before the split, since ramps don't get their own AADT) and giving them a fixed `'Ramp'` tier, then merging everything into `highways_classified`.

In [37]:
where_clause = "segment_type = 'Ramp'"
arcpy.management.SelectLayerByAttribute("highways_filtered_v2", "NEW_SELECTION", where_clause)
arcpy.management.CopyFeatures("highways_filtered_v2", "highways_ramps_only")

arcpy.management.AddField("highways_ramps_only", "traffic_tier", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_ramps_only", "traffic_tier", "'Ramp'")

arcpy.management.Merge(
    inputs=["highways_with_aadt_v2", "highways_ramps_only"],
    output="highways_classified"
)

count = arcpy.management.GetCount("highways_classified")
print(count)

5237


Symbology in ArcGIS Pro wasn't showing `traffic_tier` as an option, so I'm double-checking the field actually exists on this layer and wasn't dropped or renamed somewhere in the merge.

In [38]:
fields = arcpy.ListFields("highways_classified")
for f in fields:
    print(f.name, f.type)

OBJECTID OID
Shape Geometry
Join_Count Integer
TARGET_FID Integer
ROAD_NET_ELEMENT_ID Integer
FULL_STREET_NAME String
ABBREVIATED_STREET_NAME String
ALT_STREET_NAME String
DIRECTIONAL_PREFIX String
STREET_TYPE_PREFIX String
STREET_NAME_BODY String
STREET_TYPE_SUFFIX String
DIRECTIONAL_SUFFIX String
ROAD_CLASS String
ROAD_ELEMENT_TYPE String
L_FIRST_HOUSE_NUMBER Integer
L_LAST_HOUSE_NUMBER Integer
L_HOUSE_NUMBER_STRUCTURE String
L_STANDARD_MUNICIPALITY String
R_FIRST_HOUSE_NUMBER Integer
R_LAST_HOUSE_NUMBER Integer
R_HOUSE_NUMBER_STRUCTURE String
R_STANDARD_MUNICIPALITY String
DIRECTION_OF_TRAFFIC_FLOW String
SPEED_LIMIT Integer
NUMBER_OF_LANES Integer
PAVEMENT_STATUS String
JURISDICTION String
ROUTE_NUMBER String
SHIELD_TYPE String
GEOMETRY_UPDATE_DATETIME Date
EFFECTIVE_DATETIME Date
LENGTH Double
segment_type String
ORIG_FID Integer
ORIG_SEQ Integer
MTO_REGION String
MTO_DISTRI String
OPP_DISTRI String
COUNTY Integer
HWY String
LHRS Integer
LENGTH_1 Double
OFFSET Integer
DESCRIPTIO S

First attempt at clearing a stray selection (the map had a bright cyan highlight left over from all my `SelectLayerByAttribute` calls). This crashed — `SELECTIONSET` isn't a valid property name in this arcpy version.

In [39]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

for lyr in m.listLayers():
    if lyr.supports("SELECTIONSET"):
        arcpy.management.SelectLayerByAttribute(lyr, "CLEAR_SELECTION")

<class 'ValueError'>: Invalid value for layer_property: 'SELECTIONSET' (choices are: ['BRIGHTNESS', 'CONNECTIONPROPERTIES', 'CONTRAST', 'DATASOURCE', 'DEFINITIONQUERY', 'IS3DLAYER', 'ISBASEMAPLAYER', 'ISBROKEN', 'ISFEATURELAYER', 'ISGRAPHICSLAYER', 'ISGROUPLAYER', 'ISNETWORKANALYSTLAYER', 'ISNETWORKDATASETLAYER', 'ISPARCELFABRICLAYER', 'ISRASTERLAYER', 'ISSCENELAYER', 'ISTIMEENABLED', 'ISWEBLAYER', 'LONGNAME', 'MAXTHRESHOLD', 'METADATA', 'MINTHRESHOLD', 'NAME', 'SHOWLABELS', 'SYMBOLOGY', 'TRANSPARENCY', 'TIME', 'URI', 'VISIBLE'])

**Fixed version** — just try clearing selection on every feature layer and skip whichever ones don't support it, instead of checking a property that doesn't actually exist.

In [40]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

for lyr in m.listLayers():
    if lyr.isFeatureLayer:
        try:
            arcpy.management.SelectLayerByAttribute(lyr, "CLEAR_SELECTION")
        except Exception as e:
            print(f"Skipped {lyr.name}: {e}")

Realized partway through symbolizing that I'd completely forgotten Highway 427. Checking how it's coded in the AADT data before rebuilding anything — turned out to be simple, just `427` directly, no hidden code like QEW had.

In [41]:
matches = aadt_df[aadt_df["Location Description"].str.contains("427", na=False)]
print(matches[["Hwy No", "Location Description"]].drop_duplicates().head(10))

# Also check directly
subset = aadt_df[aadt_df["Hwy No"] == 427]
print(subset["Location Description"].drop_duplicates().head(8).tolist())

       Hwy No                  Location Description
1823        1  HWY 427 IC-139 ETOBICOKE START OF NA
22581      27                     HWY 427/401 RAMPS
39767     401                        HWY 427 IC-352
['COULES CT ETOBICOKE START OF NA', 'EVANS AV IC ETOBICOKE', 'QEW IC START OF COMPLEX FREEWAY', 'HWY 5 IC DUNDAS ST ETOBICOKE', 'BURNHAMTHORPE RD IC ETOBICOKE', 'RATHBURN RD IC ETOBICOKE', 'HWY 401 IC', 'DIXON RD IC ETOBICOKE']


Rebuilding the highway pull with 427 added to the target list, same two-pass mainline+ramp approach as before.

In [42]:
url = "https://services1.arcgis.com/TJH5KDher0W13Kgo/arcgis/rest/services/Ontario_Road_Network_Composite_Service_GeoHub_View_EN/FeatureServer/5"

target_routes = {"400", "401", "403", "404", "410", "427", "QEW"}

like_clause = " OR ".join([f"ROUTE_NUMBER LIKE '%{r}%'" for r in target_routes])
where_clause = f"({like_clause}) AND ROAD_CLASS = 'Freeway'"
arcpy.conversion.FeatureClassToFeatureClass(url, arcpy.env.workspace, "highways_mainline", where_clause)

street_names = ["HIGHWAY 400", "HIGHWAY 401", "HIGHWAY 403", "HIGHWAY 404", "HIGHWAY 410", "HIGHWAY 427", "QUEEN ELIZABETH WAY"]
street_clause = " OR ".join([f"FULL_STREET_NAME LIKE '%{s}%'" for s in street_names])
where_clause = f"({street_clause}) AND ROAD_CLASS = 'Ramp'"
arcpy.conversion.FeatureClassToFeatureClass(url, arcpy.env.workspace, "highways_ramps", where_clause)

arcpy.management.AddField("highways_mainline", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_mainline", "segment_type", "'Mainline'")
arcpy.management.AddField("highways_ramps", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_ramps", "segment_type", "'Ramp'")

arcpy.management.Merge(["highways_mainline", "highways_ramps"], "highways_filtered_v2")
print(arcpy.management.GetCount("highways_filtered_v2"))

4785


Rebuilding the AADT filtering with 427's code added in too.

In [43]:
target_hwy_codes = [1, 400, 401, 403, 404, 410, 427]  # 1 = QEW
aadt_filtered = aadt_df[aadt_df["Hwy No"].isin(target_hwy_codes)].copy()
aadt_2021 = aadt_filtered[aadt_filtered["Year"] == 2021].copy()
aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)
print(aadt_2021.shape)

(338, 30)


Re-running the cell above — looks like I ran it twice, probably just re-executing after a small tweak.

In [44]:
url = "https://services1.arcgis.com/TJH5KDher0W13Kgo/arcgis/rest/services/Ontario_Road_Network_Composite_Service_GeoHub_View_EN/FeatureServer/5"

target_routes = {"400", "401", "403", "404", "410", "427", "QEW"}

like_clause = " OR ".join([f"ROUTE_NUMBER LIKE '%{r}%'" for r in target_routes])
where_clause = f"({like_clause}) AND ROAD_CLASS = 'Freeway'"
arcpy.conversion.FeatureClassToFeatureClass(url, arcpy.env.workspace, "highways_mainline", where_clause)

street_names = ["HIGHWAY 400", "HIGHWAY 401", "HIGHWAY 403", "HIGHWAY 404", "HIGHWAY 410", "HIGHWAY 427", "QUEEN ELIZABETH WAY"]
street_clause = " OR ".join([f"FULL_STREET_NAME LIKE '%{s}%'" for s in street_names])
where_clause = f"({street_clause}) AND ROAD_CLASS = 'Ramp'"
arcpy.conversion.FeatureClassToFeatureClass(url, arcpy.env.workspace, "highways_ramps", where_clause)

arcpy.management.AddField("highways_mainline", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_mainline", "segment_type", "'Mainline'")
arcpy.management.AddField("highways_ramps", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_ramps", "segment_type", "'Ramp'")

arcpy.management.Merge(["highways_mainline", "highways_ramps"], "highways_filtered_v2")
print(arcpy.management.GetCount("highways_filtered_v2"))

4785


Same as above, a re-run of the AADT rebuild with 427 included.

In [45]:
target_hwy_codes = [1, 400, 401, 403, 404, 410, 427]  # 1 = QEW
aadt_filtered = aadt_df[aadt_df["Hwy No"].isin(target_hwy_codes)].copy()
aadt_2021 = aadt_filtered[aadt_filtered["Year"] == 2021].copy()
aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)
print(aadt_2021.shape)

aadt_lookup = {(row["LHRS"], row["Offset"]): row["AADT_clean"] for _, row in aadt_2021.iterrows()}

# Re-add the field fresh (in case it still has old values from before)
arcpy.management.AddField("lhrs_base_points", "AADT_2021_v2", "DOUBLE")
with arcpy.da.UpdateCursor("lhrs_base_points", ["LHRS", "OFFSET", "AADT_2021_v2"]) as cursor:
    for row in cursor:
        key = (row[0], row[1])
        if key in aadt_lookup:
            row[2] = aadt_lookup[key]
            cursor.updateRow(row)

where_clause = "HWY IN ('1','400','401','403','404','410','427') AND AADT_2021_v2 IS NOT NULL"
arcpy.management.SelectLayerByAttribute("lhrs_base_points", "NEW_SELECTION", where_clause)
arcpy.management.CopyFeatures("lhrs_base_points", "aadt_stations_2021_v2")
print(arcpy.management.GetCount("aadt_stations_2021_v2"))

(338, 30)
330


Rebuilding the split-at-stations step with the 427-inclusive data — this produced `highways_split` and the widened spatial join in one go this time, since I already knew the 50km radius fix from before.

In [46]:
arcpy.management.SelectLayerByAttribute("highways_filtered_v2", "NEW_SELECTION", "segment_type = 'Mainline'")
arcpy.management.CopyFeatures("highways_filtered_v2", "highways_mainline_only")

arcpy.management.SplitLineAtPoint(
    in_features="highways_mainline_only",
    point_features="aadt_stations_2021_v2",
    out_feature_class="highways_split",
    search_radius="100 Meters"
)
print(arcpy.management.GetCount("highways_split"))

arcpy.analysis.SpatialJoin(
    target_features="highways_split",
    join_features="aadt_stations_2021_v2",
    out_feature_class="highways_with_aadt_v2",
    join_operation="JOIN_ONE_TO_ONE",
    match_option="CLOSEST",
    search_radius="50000 Meters"
)
print(arcpy.management.GetCount("highways_with_aadt_v2"))

with arcpy.da.SearchCursor("highways_with_aadt_v2", ["AADT_2021_v2"]) as cursor:
    null_count = sum(1 for row in cursor if row[0] is None)
print(f"Segments with no matched AADT: {null_count}")

2945
2945
Segments with no matched AADT: 0


Recomputing the tercile break points now that 427's traffic numbers are mixed into the pool (they shifted slightly, as expected), and reclassifying every stretch.

In [47]:
with arcpy.da.SearchCursor("highways_with_aadt_v2", ["AADT_2021_v2"]) as cursor:
    values = [row[0] for row in cursor]

import numpy as np
values = np.array(values)

low_threshold = np.percentile(values, 33.33)
high_threshold = np.percentile(values, 66.67)
print(f"Low/Medium boundary: {low_threshold:.0f}")
print(f"Medium/High boundary: {high_threshold:.0f}")

arcpy.management.AddField("highways_with_aadt_v2", "traffic_tier", "TEXT", field_length=20)
with arcpy.da.UpdateCursor("highways_with_aadt_v2", ["AADT_2021_v2", "traffic_tier"]) as cursor:
    for row in cursor:
        aadt = row[0]
        if aadt is None:
            row[1] = "No Data"
        elif aadt <= low_threshold:
            row[1] = "Low"
        elif aadt <= high_threshold:
            row[1] = "Medium"
        else:
            row[1] = "High"
        cursor.updateRow(row)

Low/Medium boundary: 41200
Medium/High boundary: 143200


Merging the 427-inclusive ramps back in to build the final `highways_classified_v2` layer — this one supersedes everything from before 427 was added.

In [48]:
arcpy.management.SelectLayerByAttribute("highways_filtered_v2", "NEW_SELECTION", "segment_type = 'Ramp'")
arcpy.management.CopyFeatures("highways_filtered_v2", "highways_ramps_only")
arcpy.management.AddField("highways_ramps_only", "traffic_tier", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_ramps_only", "traffic_tier", "'Ramp'")

arcpy.management.Merge(["highways_with_aadt_v2", "highways_ramps_only"], "highways_classified_v2")
print(arcpy.management.GetCount("highways_classified_v2"))

5539


Clearing selection again before jumping into symbology, same fix as cell 48, just making sure nothing's still highlighted cyan.

In [49]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap
for lyr in m.listLayers():
    if lyr.isFeatureLayer:
        try:
            arcpy.management.SelectLayerByAttribute(lyr, "CLEAR_SELECTION")
        except Exception as e:
            pass

Listing every layer currently sitting in my map's Contents pane — by this point there were about 14 scratch/intermediate layers cluttering things up, so I wanted to see exactly what needed cleaning out.

In [50]:
import arcpy

aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

# Layer names to keep visible - everything else gets removed from the map
keep_layers = ["highways_classified_v2"]

# List what's currently in the map first, so you can see what will be removed
for lyr in m.listLayers():
    print(lyr.name)

aadt_stations_2021_v2
aadt_stations_2021
lhrs_base_points
highways_classified_v2
highways_ramps_only
highways_with_aadt_v2
highways_split
highways_mainline_only
highways_filtered_v2
highways_ramps
highways_mainline
highways_classified
highways_with_aadt
check_403_gap
highways_filtered
lhrs_routes
World Topographic Map
World Hillshade


**Actually cleaning up** — removing everything except my final classified layer and the two basemap layers. This only touches the map view, not the underlying geodatabase, so nothing's actually deleted.

In [51]:
keep_layers = ["highways_classified_v2", "World Topographic Map", "World Hillshade"]

for lyr in m.listLayers():
    if lyr.name not in keep_layers:
        m.removeLayer(lyr)

# Confirm what's left
for lyr in m.listLayers():
    print(lyr.name)
    

highways_classified_v2
World Topographic Map
World Hillshade


Checking the coordinate system before exporting to a web map — web maps need plain lat/long (WGS84), and I wanted to confirm what I was starting from.

In [52]:
desc = arcpy.Describe("highways_classified_v2")
print(desc.spatialReference.name)

GCS_North_American_1983


Reprojecting to WGS84 so the data will actually line up correctly once it hits Leaflet in the browser.

In [53]:
arcpy.management.Project(
    in_dataset="highways_classified_v2",
    out_dataset="highways_classified_wgs84",
    out_coor_system=arcpy.SpatialReference(4326)  # WGS84
)

<Result 'C:\\Users\\ajots\\OneDrive\\Documents\\ArcGIS\\Projects\\GTA_Highway_Traffic_Volumes\\GTA_Highway_Traffic_Volumes.gdb\\highways_classified_wgs84'>

Exporting the classified, reprojected layer to GeoJSON — this is the file the actual web map reads. (Note: first export went into `data raw` by mistake — moved to `data processed` afterward for the real workflow.)

In [54]:
arcpy.conversion.FeaturesToJSON(
    in_features="highways_classified_wgs84",
    out_json_file=r"data raw\highways_classified.geojson",
    geoJSON="GEOJSON"
)

<Result 'data raw\\highways_classified.geojson'>

First attempt at merging the GeoJSON into the HTML template as embedded data, so the map works via double-click with no local server. This version had a bug — more on that below.

In [55]:
import json

# Read your GeoJSON
with open(r"data processed\highways_classified.geojson", "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

# Read the HTML template
with open(r"data processed\highways_classified.html", "r", encoding="utf-8") as f:
    html_content = f.read()

# Build the embedded data script tag
embed_script = f"<script>window.EMBEDDED_HIGHWAY_DATA = {json.dumps(geojson_data)};</script>\n"

# Insert it right before the closing </body> tag
merged_html = html_content.replace("</body>", embed_script + "</body>")

# Save as a new standalone file
with open(r"data processed\highways_classified_standalone.html", "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Done - standalone file created")

Done - standalone file created


Re-running basically the same merge attempt — still hadn't found the actual bug yet at this point.

In [56]:
import json

with open(r"data processed\highways_classified.geojson", "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

with open(r"data processed\highways_classified.html", "r", encoding="utf-8") as f:
    html_content = f.read()

embed_script = f"<script>window.EMBEDDED_HIGHWAY_DATA = {json.dumps(geojson_data)};</script>\n"
merged_html = html_content.replace("</body>", embed_script + "</body>")

with open(r"data processed\highways_classified_standalone.html", "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Done - standalone file created")

<class 'FileNotFoundError'>: [Errno 2] No such file or directory: 'data processed\\highways_classified.html'

Switching to the reliable absolute-path pattern (same trick as cell 29) since a plain relative path failed to find the HTML template file.

In [57]:
import json
import arcpy
import os

aprx = arcpy.mp.ArcGISProject("CURRENT")
project_folder = aprx.homeFolder

geojson_path = os.path.join(project_folder, "data processed", "highways_classified.geojson")
html_path = os.path.join(project_folder, "data processed", "highways_classified.html")
output_path = os.path.join(project_folder, "data processed", "highways_classified_standalone.html")

print(os.path.exists(geojson_path))
print(os.path.exists(html_path))

True
False


Re-running the path setup above — this is where I discovered the HTML template genuinely hadn't been moved into the right folder yet.

In [58]:
import json
import arcpy
import os

aprx = arcpy.mp.ArcGISProject("CURRENT")
project_folder = aprx.homeFolder

geojson_path = os.path.join(project_folder, "data processed", "highways_classified.geojson")
html_path = os.path.join(project_folder, "data processed", "highways_classified.html")
output_path = os.path.join(project_folder, "data processed", "highways_classified_standalone.html")

print(os.path.exists(geojson_path))
print(os.path.exists(html_path))

True
True


**The actual embedding bug.** I was inserting the embedded data right before `</body>`, but the map's own logic script runs earlier in the file and checks for that data *before* it exists yet — so the check always failed and fell through to a broken fetch. This is what caused the "could not load" error even with embedded data.

In [59]:
with open(geojson_path, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

embed_script = f"<script>window.EMBEDDED_HIGHWAY_DATA = {json.dumps(geojson_data)};</script>\n"
merged_html = html_content.replace("</body>", embed_script + "</body>")

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Done - standalone file created at:", output_path)

Done - standalone file created at: C:\Users\ajots\OneDrive\Documents\ArcGIS\Projects\GTA_Highway_Traffic_Volumes\data processed\highways_classified_standalone.html


**Real fix** — inserting the embedded data before the Leaflet script tag instead, so it's defined *before* anything tries to read it. Order matters when you're injecting data into an HTML file like this.

In [60]:
with open(geojson_path, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

embed_script = f"<script>window.EMBEDDED_HIGHWAY_DATA = {json.dumps(geojson_data)};</script>\n"

# Insert BEFORE the Leaflet script tag, not before </body> - order matters!
insertion_point = '<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>'
merged_html = html_content.replace(insertion_point, embed_script + insertion_point)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Done - standalone file created at:", output_path)

Done - standalone file created at: C:\Users\ajots\OneDrive\Documents\ArcGIS\Projects\GTA_Highway_Traffic_Volumes\data processed\highways_classified_standalone.html


Checking every field on `highways_classified_v2` before building the "Highway Overview" feature — specifically confirming `L_STANDARD_MUNICIPALITY` / `R_STANDARD_MUNICIPALITY` survived the whole pipeline, since that's what makes "cities this highway runs through" possible at all.

In [61]:
fields = arcpy.ListFields("highways_classified_v2")
for f in fields:
    print(f.name, f.type)

OBJECTID OID
Shape Geometry
Join_Count Integer
TARGET_FID Integer
ROAD_NET_ELEMENT_ID Integer
FULL_STREET_NAME String
ABBREVIATED_STREET_NAME String
ALT_STREET_NAME String
DIRECTIONAL_PREFIX String
STREET_TYPE_PREFIX String
STREET_NAME_BODY String
STREET_TYPE_SUFFIX String
DIRECTIONAL_SUFFIX String
ROAD_CLASS String
ROAD_ELEMENT_TYPE String
L_FIRST_HOUSE_NUMBER Integer
L_LAST_HOUSE_NUMBER Integer
L_HOUSE_NUMBER_STRUCTURE String
L_STANDARD_MUNICIPALITY String
R_FIRST_HOUSE_NUMBER Integer
R_LAST_HOUSE_NUMBER Integer
R_HOUSE_NUMBER_STRUCTURE String
R_STANDARD_MUNICIPALITY String
DIRECTION_OF_TRAFFIC_FLOW String
SPEED_LIMIT Integer
NUMBER_OF_LANES Integer
PAVEMENT_STATUS String
JURISDICTION String
ROUTE_NUMBER String
SHIELD_TYPE String
GEOMETRY_UPDATE_DATETIME Date
EFFECTIVE_DATETIME Date
LENGTH Double
segment_type String
ORIG_FID Integer
ORIG_SEQ Integer
MTO_REGION String
MTO_DISTRI String
OPP_DISTRI String
COUNTY Integer
HWY String
LHRS Integer
LENGTH_1 Double
OFFSET Integer
DESCRIPTIO S

**Building the per-highway stats** for the overview panel — cities, average AADT, busiest/quietest points, interchange counts, and (buggy, at this point) total length. Used the original `LENGTH` field here, which turned out to be stale after all my splitting.

In [62]:
import json

target_highways = ["400", "401", "403", "404", "410", "427", "QEW"]
highway_stats = {}

with arcpy.da.SearchCursor("highways_classified_v2",
    ["ROUTE_NUMBER", "segment_type", "L_STANDARD_MUNICIPALITY", "R_STANDARD_MUNICIPALITY",
     "LENGTH", "AADT_2021_v2", "DESCRIPTIO", "FULL_STREET_NAME"]) as cursor:
    rows = list(cursor)

for hwy in target_highways:
    cities = set()
    total_length = 0.0
    aadt_values = []
    busiest = {"aadt": -1, "desc": None}
    quietest = {"aadt": float("inf"), "desc": None}
    ramp_count = 0

    for r in rows:
        route_num, seg_type, l_muni, r_muni, length, aadt, desc, street = r

        # Mainline stats: match if this highway appears in the (possibly combined) ROUTE_NUMBER
        if seg_type == "Mainline" and route_num and hwy in [x.strip() for x in route_num.split(";")]:
            if l_muni: cities.add(l_muni)
            if r_muni: cities.add(r_muni)
            if length: total_length += length
            if aadt is not None:
                aadt_values.append(aadt)
                if aadt > busiest["aadt"]:
                    busiest = {"aadt": aadt, "desc": desc}
                if aadt < quietest["aadt"]:
                    quietest = {"aadt": aadt, "desc": desc}

        # Ramp count: match if this highway's name appears in the ramp's street name
        if seg_type == "Ramp" and street:
            street_upper = street.upper()
            hwy_label = "QUEEN ELIZABETH WAY" if hwy == "QEW" else f"HIGHWAY {hwy}"
            if hwy_label in street_upper:
                ramp_count += 1

    highway_stats[hwy] = {
        "cities": sorted(cities),
        "total_length_km": round(total_length, 1),
        "avg_aadt": round(sum(aadt_values) / len(aadt_values)) if aadt_values else None,
        "busiest": busiest,
        "quietest": quietest,
        "interchange_count": ramp_count
    }

for hwy, stats in highway_stats.items():
    print(hwy, "->", stats)

400 -> {'cities': ['City of Barrie', 'City of Toronto', 'City of Vaughan', 'Town of Bradford West Gwillimbury', 'Town of Innisfil', 'Town of Parry Sound', 'Township of Georgian Bay', 'Township of King', 'Township of McDougall', 'Township of Oro-Medonte', 'Township of Seguin', 'Township of Severn', 'Township of Springwater', 'Township of Tay', 'Wahta Mohawk Territory'], 'total_length_km': 562567.1, 'avg_aadt': 77823, 'busiest': {'aadt': 424600.0, 'desc': 'HWY 400 IC-359-NORTH YORK'}, 'quietest': {'aadt': 8100.0, 'desc': 'GEORGIAN BAY RD - CROOKED BAY RD IC 168'}, 'interchange_count': 356}
401 -> {'cities': ['City of Belleville', 'City of Brockville', 'City of Cambridge', 'City of Cornwall', 'City of Kingston', 'City of London', 'City of Mississauga', 'City of Oshawa', 'City of Pickering', 'City of Quinte West', 'City of Toronto', 'City of Windsor', 'City of Woodstock', 'Municipality of Brighton', 'Municipality of Chatham-Kent', 'Municipality of Clarington', 'Municipality of Dutton/Dunwi

First attempt at fixing the length bug using `arcpy.management.CalculateGeometryAttributes` — used the wrong parameter name (`GEODESIC_LENGTH` instead of the correct `LENGTH_GEODESIC`) and it errored out immediately.

In [63]:
arcpy.management.CalculateGeometryAttributes(
    in_features="highways_classified_v2",
    geometry_property=[["seg_length_km", "GEODESIC_LENGTH"]],
    length_unit="KILOMETERS"
)

<class 'arcgisscripting.ExecuteError'>: Failed to execute. Parameters are not valid.
ERROR 000800: The value is not a member of LENGTH_GEODESIC | LINE_BEARING | PART_COUNT | POINT_COUNT | CURVE_COUNT | LINE_START_X | LINE_START_Y | LINE_START_M | LINE_END_X | LINE_END_Y | LINE_END_M | CENTROID_X | CENTROID_Y | CENTROID_M | INSIDE_X | INSIDE_Y | INSIDE_M | EXTENT_MIN_X | EXTENT_MIN_Y | EXTENT_MAX_X | EXTENT_MAX_Y.
Failed to execute (CalculateGeometryAttributes).


**Corrected parameter name.** This actually recalculates each stretch's real current length straight from its geometry, instead of relying on the stale pre-split `LENGTH` attribute that was massively overcounting everything.

In [64]:
arcpy.management.CalculateGeometryAttributes(
    in_features="highways_classified_v2",
    geometry_property=[["seg_length_km", "LENGTH_GEODESIC"]],
    length_unit="KILOMETERS"
)

<Result 'highways_classified_v2'>

Re-running the total length aggregation using the corrected `seg_length_km` field. Numbers were still roughly double real-world figures at this point — see the next two cells for why.

In [65]:
with arcpy.da.SearchCursor("highways_classified_v2",
    ["ROUTE_NUMBER", "segment_type", "seg_length_km"]) as cursor:
    rows = list(cursor)

for hwy in target_highways:
    total_length = 0.0
    for route_num, seg_type, seg_len in rows:
        if seg_type == "Mainline" and route_num and hwy in [x.strip() for x in route_num.split(";")]:
            if seg_len:
                total_length += seg_len
    highway_stats[hwy]["total_length_km"] = round(total_length, 1)

for hwy in target_highways:
    print(hwy, highway_stats[hwy]["total_length_km"])

400 452.4
401 1784.2
403 253.8
404 100.9
410 43.8
427 58.4
QEW 278.4


Checking `DIRECTION_OF_TRAFFIC_FLOW` to understand the doubling — turns out divided highways are digitized as two separate carriageway lines (one per direction), which is exactly why summing length was giving ~2x the real distance.

In [66]:
with arcpy.da.SearchCursor("highways_classified_v2", ["DIRECTION_OF_TRAFFIC_FLOW"]) as cursor:
    from collections import Counter
    tally = Counter(row[0] for row in cursor)

for val, n in tally.items():
    print(repr(val), n)

'Negative' 2759
'Positive' 2740
'Both' 40


Halving the total length figures to correct for the double-digitized carriageways — landed close to real-world highway lengths after this (QEW's 139km matched almost exactly).

In [67]:
for hwy in target_highways:
    highway_stats[hwy]["total_length_km"] = round(highway_stats[hwy]["total_length_km"] / 2, 1)

for hwy in target_highways:
    print(hwy, highway_stats[hwy]["total_length_km"])

400 226.2
401 892.1
403 126.9
404 50.5
410 21.9
427 29.2
QEW 139.2


**Final build.** Embedding both the GeoJSON map data and the highway_stats dictionary into the HTML template, in the correct order this time, to produce the final standalone `highways_classified_standalone.html` — one self-contained file with the whole interactive map and the Highway Overview panel baked in.

In [68]:
import json
import arcpy
import os

aprx = arcpy.mp.ArcGISProject("CURRENT")
project_folder = aprx.homeFolder

geojson_path = os.path.join(project_folder, "data processed", "highways_classified.geojson")
html_path = os.path.join(project_folder, "data processed", "highways_classified.html")  # the NEW download from above
output_path = os.path.join(project_folder, "data processed", "highways_classified_standalone.html")

with open(geojson_path, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

geojson_embed = f"<script>window.EMBEDDED_HIGHWAY_DATA = {json.dumps(geojson_data)};</script>\n"
stats_embed = f"<script>window.HIGHWAY_STATS = {json.dumps(highway_stats)};</script>\n"

insertion_point = '<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>'
merged_html = html_content.replace(insertion_point, geojson_embed + stats_embed + insertion_point)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Done:", output_path)

Done: C:\Users\ajots\OneDrive\Documents\ArcGIS\Projects\GTA_Highway_Traffic_Volumes\data processed\highways_classified_standalone.html


Empty trailing cell — nothing here.